# Classification of q-matroids

- Enumerate the isomorphism classes of q-matroids using the subspace order defined in the paper and one-dimensional extensions.
- Save a canonical representative of each isomorphism class, grouped by dimension and rank.

In [1]:
import os
import subprocess
import sys
from datetime import timedelta
from pathlib import Path
from time import perf_counter

from sage.all import table
from sage.version import version as sage_version

from qmatroid.finite_geometry import get_subspace_tables
from qmatroid.q_matroid_enumeration import (
    enumerate_q_matroids,
    save_q_matroids,
)

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

## Settings

- `q`: The order of the field.
- `N`: The maximum dimension of the ambient spaces to classify.
  - The default settings `q = 2`, `N = 4` classify dimensions up to 4 over $\mathrm{GF}(2)$.
  - To classify dimensions up to 5 over $\mathrm{GF}(2)$, set `N = 5`.
  - To classify dimensions up to 4 over $\mathrm{GF}(3)$, set `q = 3`, `N = 4`.
- `workers`: The number of processes used to check canonical representatives. Parallel computation requires Linux or WSL2.
- `output_directory`: The directory for saving results. One text file is created for each dimension and rank.
- Classifying rank-2 q-matroids on $\mathrm{GF}(2)^5$ may take tens of hours.

In [ ]:
q = 2
N = 5
workers = 16
case = f"q{q}_d{N}"
target_rank = N // 2
output_directory = project_root / "results" / "generated" / f"q{q}_d{N}"
target_output_path = output_directory / f"q{q}_dimension{N}_rank{target_rank}.txt"
output_directory.mkdir(parents=True, exist_ok=True)
tables = get_subspace_tables(q, N)

display(table(
    [
        ("case", case),
        ("field order", q),
        ("maximum dimension", N),
        ("long target rank", target_rank),
        ("workers", workers),
        ("output", str(target_output_path.relative_to(project_root))),
    ],
    header_row=["setting", "value"],
))
display(table(
    [
        ("field order", tables.q),
        ("maximum dimension", tables.max_dimension),
        ("stored subspaces", tables.number_of_subspaces),
        ("subspace order version", tables.order_version),
    ],
    header_row=["table property", "value"],
))

  setting             value
├───────────────────┼─────────────────────────────────────────────────┤
  case                q2_d5
  field order         2
  maximum dimension   5
  long target rank    2
  workers             20
  output              results/generated/q2_d5/q2_dimension5_rank2.txt

  table property           value
├────────────────────────┼───────┤
  field order              2
  maximum dimension        5
  stored subspaces         374
  subspace order version   v1

## Measuring computational resources

- Measure the wall-clock time spent on enumeration, dualization, and saving files in the classification cell.
- A separate process samples the total PSS of the Notebook kernel and its child processes (including parallel workers) approximately every 0.1 seconds.
  - PSS is a measure of memory usage that distributes shared memory proportionally among the processes sharing it.
  - This includes the memory used by SageMath and the subspace tables loaded by the kernel.
  - The maximum sampled value is recorded, so brief peaks between samples may be missed.
- After the computation finishes, the final table displays the CPU model, memory usage, and execution time.

In [12]:
cpu_model = next(
    line.split(":", 1)[1].strip()
    for line in Path("/proc/cpuinfo").read_text().splitlines()
    if line.startswith("model name")
)

memory_sampler = """
import os
import select
import sys

import psutil

parent = psutil.Process(int(sys.argv[1]))
peak = parent.memory_full_info().pss
print(peak, flush=True)
while True:
    total = 0
    for process in [parent, *parent.children(recursive=True)]:
        if process.pid == os.getpid():
            continue
        try:
            total += process.memory_full_info().pss
        except psutil.NoSuchProcess:
            pass
    peak = max(peak, total)
    if select.select([sys.stdin], [], [], 0.1)[0]:
        break
print(peak)
"""

## Direct enumeration and duality

- $n$ is the dimension of the ambient space, and $k$ is the rank of the q-matroid.
- $k\leq n/2$: Enumerate canonical representatives using one-dimensional extensions.
- $k>n/2$: Take the duals of the rank-$n-k$ canonical representatives and find their canonical representatives with respect to the fixed subspace order.

In [13]:
memory_monitor = subprocess.Popen(
    [sys.executable, "-c", memory_sampler, str(os.getpid())],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    text=True,
    start_new_session=True,
)
peak_memory_bytes = int(memory_monitor.stdout.readline())
classification_started = perf_counter()
try:
    classification = {}

    for n in range(N + 1):
        for k in range(n // 2 + 1):
            started = perf_counter()
            classification[n, k] = tuple(
                enumerate_q_matroids(
                    q, n, k, tables=tables, workers=workers
                )
            )
            path = output_directory / f"q{q}_dimension{n}_rank{k}.txt"
            save_q_matroids(
                sorted(classification[n, k], key=lambda M: M.basis_encoding(tables)),
                path,
            )
            print(
                f"dimension={n}, rank={k}: "
                f"{len(classification[n, k])} representatives "
                f"({perf_counter() - started:.2f} seconds)"
            )

        for k in range(n // 2 + 1, n + 1):
            classification[n, k] = tuple(
                M.dual(tables).canonical_representative(tables)
                for M in classification[n, n - k]
            )
            path = output_directory / f"q{q}_dimension{n}_rank{k}.txt"
            save_q_matroids(
                sorted(classification[n, k], key=lambda M: M.basis_encoding(tables)),
                path,
            )
finally:
    elapsed_seconds = perf_counter() - classification_started
    peak_memory_bytes = int(memory_monitor.communicate("\n")[0])

dimension=0, rank=0: 1 representatives (0.00 seconds)
dimension=1, rank=0: 1 representatives (0.16 seconds)
dimension=2, rank=0: 1 representatives (0.26 seconds)
dimension=2, rank=1: 2 representatives (0.40 seconds)
dimension=3, rank=0: 1 representatives (0.38 seconds)
dimension=3, rank=1: 3 representatives (0.68 seconds)
dimension=4, rank=0: 1 representatives (0.50 seconds)
dimension=4, rank=1: 4 representatives (0.94 seconds)
dimension=4, rank=2: 10 representatives (1.16 seconds)
dimension=5, rank=0: 1 representatives (0.65 seconds)
dimension=5, rank=1: 5 representatives (1.29 seconds)
dimension=5, rank=2: 94 representatives (5955.22 seconds)


## Number of isomorphism classes

- Display the number of isomorphism classes for each dimension and rank, together with the total for each dimension.
- Finally, display the directory where the results were saved.

In [ ]:
rows = []
for n in range(N + 1):
    counts = tuple(len(classification[n, k]) for k in range(n + 1))
    rows.append(
        (n, *counts, *("" for _ in range(N - n)), sum(counts))
    )

display(table(
    rows,
    header_row=[
        "dimension",
        *(f"rank {k}" for k in range(N + 1)),
        "total",
    ],
))
print(f"Generated files are in {output_directory.relative_to(project_root)}.")

  dimension   rank 0   rank 1   rank 2   rank 3   rank 4   rank 5   total
├───────────┼────────┼────────┼────────┼────────┼────────┼────────┼───────┤
  0           1                                                     1
  1           1        1                                            2
  2           1        2        1                                   4
  3           1        3        3        1                          8
  4           1        4        10       4        1                 20
  5           1        5        94       94       5        1        200

Generated files are in results/generated/q2_d5.


## Computational resources used

- The CPU model is obtained from the execution environment.
- `workers` is the number of parallel workers specified for checking canonical representatives.
- Peak memory usage is the maximum total PSS of the kernel and its child processes observed during classification (in GiB).
- Execution time is the wall-clock time from the start of enumeration until all dualization and saving are complete.

In [15]:
display(table(
    [
        ("CPU model", cpu_model),
        ("available logical CPUs", len(os.sched_getaffinity(0))),
        ("workers", workers),
        ("peak total PSS (GiB, sampled)", f"{peak_memory_bytes / 1024**3:.3f}"),
        ("elapsed time (h:mm:ss)", str(timedelta(seconds=round(elapsed_seconds)))),
        ("elapsed time (seconds)", f"{elapsed_seconds:.3f}"),
    ],
    header_row=["resource", "value"],
))

  resource                        value
├───────────────────────────────┼───────────────────────────────────────┤
  CPU model                       13th Gen Intel(R) Core(TM) i9-13980HX
  available logical CPUs          32
  workers                         20
  peak total PSS (GiB, sampled)   2.650
  elapsed time (h:mm:ss)          1:39:23
  elapsed time (seconds)          5963.325